# vLLM 吞吐与前缀复用 —— 云端受控复现

在 **Colab T4** 上跑。相对本地 GTX 1650 的两点关键差异：

| | GTX 1650（本地） | T4（本 notebook） |
|---|---|---|
| 架构 | sm_75 | sm_75（同代） |
| **Tensor Core** | **无** | **有** |
| 显存 | 4 GiB | 16 GiB |

**目的一**：本地 fp16 出现 batch≥2 吞吐塌陷（并发 4 时 fp32 快 9.1×）。
T4 若无此现象，则「无 Tensor Core 导致」再得一个独立佐证。

**目的二**：本地那组前缀复用数据因处在塌陷状态而**判定无效**。
T4 上 fp16 正常，可做 `--enable-prefix-caching` 开/关的受控对照，得出真实收益。


## 0. 环境


In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"],capture_output=True,text=True).stdout)
p=torch.cuda.get_device_properties(0)
print(f"{p.name} sm_{p.major}{p.minor} {p.total_memory/2**30:.1f} GiB")
print("Tensor Core:", "无（GTX 16 系）" if "GTX 16" in p.name else "有")


## 1. 装 vLLM（几分钟）

两个 Colab 上必踩的坑，本格已处理：

1. **`libcudart.so.13: cannot open shared object file`** —— vLLM 会把 torch 升到 cu13，
   但会话里已加载的是 cu12，动态库路径要重启才重新解析。本格装完会**自动重启运行时**。
2. **`PyTorch has CUDA 13.0 whereas TorchAudio has CUDA 12.8`** —— Colab 预装的
   torchaudio / torchvision 是 cu12.8，不随 vLLM 升级；transformers 的导入链会碰到它，
   导致 **服务进程静默崩溃、压测只报连接失败**。纯文本模型用不上它们，直接卸掉。

> ⚠️ **本格会自动重启运行时。重启后请再点一次「全部运行」**，
> 届时本格检测到 vLLM 已装会直接跳过，几秒内通过。


In [ ]:
import importlib.util, subprocess, sys, os

if importlib.util.find_spec("vllm") is None:
    print("首次安装 vLLM …")
    subprocess.run([sys.executable,"-m","pip","install","-q","vllm","aiohttp"],check=False)
    # Colab 预装的 torchaudio / torchvision 是 cu12.8，与 vLLM 带来的 cu13 torch 不匹配，
    # transformers 导入链会触发 RuntimeError 并让 vllm serve 静默退出。纯文本模型不需要它们。
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","torchaudio","torchvision"],
                   check=False)
    print("安装完成。正在自动重启运行时 —— 重启后请再点一次「全部运行」。")
    os.kill(os.getpid(), 9)          # Colab 标准做法：强制重启内核以重新解析动态库
else:
    import torch, vllm
    print("vllm ", vllm.__version__)
    print("torch", torch.__version__)
    for m in ("torchaudio","torchvision"):
        print(f"{m:<12}", "已移除（预期）" if importlib.util.find_spec(m) is None else "仍在，可能报 CUDA 版本不匹配")


## 2. 写出压测脚本


In [ ]:
import io
files = {}

files['bench_serving.py'] = r'''# -*- coding: utf-8 -*-
"""vLLM 服务端压测：并发扫描下的吞吐 / TTFT / TPOT，以及前缀复用的效果。

指标定义（与 JD 里那套一致）：
  TTFT  Time To First Token   —— 首 token 延迟，决定交互体感
  TPOT  Time Per Output Token —— 首 token 之后的平均出词间隔
  吞吐   总输出 token 数 / 墙钟时间

用法（先 bash serve.sh 起服务）：
  python bench_serving.py                 # 并发扫描
  python bench_serving.py --prefix-test   # 前缀复用对照
"""
import argparse, asyncio, json, statistics as st, time
import aiohttp

URL = "http://127.0.0.1:8000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# 一段较长的共享 system prompt：开 --enable-prefix-caching 后其 prefill 只算一次
SHARED_PREFIX = (
    "You are a meticulous technical assistant. Answer concisely and precisely. "
    "Always reason step by step before answering. " * 20
)


async def one_request(sess, prompt, max_tokens, use_prefix):
    msgs = ([{"role": "system", "content": SHARED_PREFIX}] if use_prefix else []) + \
           [{"role": "user", "content": prompt}]
    body = {"model": MODEL, "messages": msgs, "max_tokens": max_tokens,
            "temperature": 0.0, "stream": True}
    t0 = time.perf_counter()
    ttft, n_tok, last = None, 0, t0
    async with sess.post(URL, json=body) as resp:
        async for raw in resp.content:
            line = raw.decode("utf-8").strip()
            if not line.startswith("data: ") or line == "data: [DONE]":
                continue
            delta = json.loads(line[6:])["choices"][0].get("delta", {})
            if delta.get("content"):
                now = time.perf_counter()
                if ttft is None:
                    ttft = now - t0
                n_tok += 1
                last = now
    return dict(ttft=ttft or 0.0, total=last - t0, n_tok=n_tok)


async def run_batch(n_conc, n_req, max_tokens, use_prefix):
    prompts = [f"Explain concept #{i} in distributed systems." for i in range(n_req)]
    sem = asyncio.Semaphore(n_conc)

    async def guarded(sess, p):
        async with sem:
            return await one_request(sess, p, max_tokens, use_prefix)

    timeout = aiohttp.ClientTimeout(total=600)
    async with aiohttp.ClientSession(timeout=timeout) as sess:
        await one_request(sess, "warmup", 4, use_prefix)          # 预热
        t0 = time.perf_counter()
        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))
        wall = time.perf_counter() - t0

    tot_tok = sum(r["n_tok"] for r in rs)
    tpots = [(r["total"] - r["ttft"]) / max(r["n_tok"] - 1, 1) for r in rs if r["n_tok"] > 1]
    return dict(conc=n_conc, wall=wall, tput=tot_tok / wall, rps=len(rs) / wall,
                ttft_p50=st.median(r["ttft"] for r in rs),
                ttft_p99=sorted(r["ttft"] for r in rs)[int(len(rs) * 0.99) - 1],
                tpot_p50=st.median(tpots) if tpots else 0.0, tot_tok=tot_tok)


async def sweep(args):
    print(f"{'并发':>5}{'请求':>6}{'墙钟s':>9}{'吞吐 tok/s':>13}{'RPS':>8}"
          f"{'TTFT p50':>11}{'TTFT p99':>11}{'TPOT p50':>11}")
    print("-" * 74)
    out = []
    for c in [1, 2, 4, 8, 16, 32]:
        r = await run_batch(c, max(c * 4, 16), args.max_tokens, use_prefix=False)
        print(f"{r['conc']:>5}{max(c*4,16):>6}{r['wall']:>9.2f}{r['tput']:>13.1f}"
              f"{r['rps']:>8.2f}{r['ttft_p50']*1e3:>10.1f}ms{r['ttft_p99']*1e3:>10.1f}ms"
              f"{r['tpot_p50']*1e3:>10.2f}ms")
        out.append(r)
    json.dump(out, open("sweep_results.json", "w"), indent=1)
    base = out[0]["tput"]
    print(f"\ncontinuous batching 收益：并发 1 → 32，吞吐 "
          f"{base:.1f} → {out[-1]['tput']:.1f} tok/s（{out[-1]['tput']/base:.1f}×），"
          f"TTFT p50 {out[0]['ttft_p50']*1e3:.0f} → {out[-1]['ttft_p50']*1e3:.0f} ms")
    print("吞吐与延迟的取舍就在这张表里：并发拉高吞吐涨，但 TTFT 同步恶化。")


async def prefix_test(args):
    print("前缀复用对照（服务端需带 --enable-prefix-caching 启动）")
    print(f"{'场景':<26}{'吞吐 tok/s':>13}{'TTFT p50':>12}")
    print("-" * 51)
    for label, up in [("无共享前缀", False), (f"共享前缀 ({len(SHARED_PREFIX)} 字符)", True)]:
        r = await run_batch(8, 32, args.max_tokens, use_prefix=up)
        print(f"{label:<26}{r['tput']:>13.1f}{r['ttft_p50']*1e3:>11.1f}ms")
    print("\n共享前缀命中 KV cache 后，重复的 prefill 不再重算，TTFT 应显著下降。")
    print("对比未开 --enable-prefix-caching 重启服务再跑一次，差值即为该特性的真实收益。")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--max-tokens", type=int, default=128)
    ap.add_argument("--prefix-test", action="store_true")
    a = ap.parse_args()
    asyncio.run(prefix_test(a) if a.prefix_test else sweep(a))
'''

files['isolate.py'] = r'''# -*- coding: utf-8 -*-
"""隔离测试：一次性发 N 条并发请求，期间无新请求到达。
用于区分「稳态 batch=N 解码慢」与「新请求 prefill 插队拖累解码」。"""
import asyncio, sys, time, json, statistics as st
import aiohttp
URL="http://127.0.0.1:8000/v1/chat/completions"; MODEL="Qwen/Qwen2.5-0.5B-Instruct"

async def one(s, i, n_tok):
    b={"model":MODEL,"messages":[{"role":"user","content":f"Explain idea {i} briefly."}],
       "max_tokens":n_tok,"temperature":0.0,"stream":True}
    t0=time.perf_counter(); ttft=None; n=0; last=t0
    async with s.post(URL,json=b) as r:
        async for raw in r.content:
            l=raw.decode().strip()
            if not l.startswith("data: ") or l=="data: [DONE]": continue
            d=json.loads(l[6:])["choices"][0].get("delta",{})
            if d.get("content"):
                now=time.perf_counter()
                if ttft is None: ttft=now-t0
                n+=1; last=now
    return (ttft or 0), last-t0, n

async def burst(n, n_tok=64):
    """严格同时发 n 条，全部跑完才结束 —— 稳态就是 batch=n。"""
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=600)) as s:
        await one(s,-1,4)                       # 预热
        t0=time.perf_counter()
        rs=await asyncio.gather(*(one(s,i,n_tok) for i in range(n)))
        wall=time.perf_counter()-t0
    tp=[(tot-tt)/max(k-1,1) for tt,tot,k in rs if k>1]
    tot=sum(k for _,_,k in rs)
    print(f"  一次性 {n:>2} 条并发: 墙钟 {wall:>6.2f}s  总吞吐 {tot/wall:>6.1f} tok/s  "
          f"TPOT p50 {st.median(tp)*1e3:>7.2f} ms")

async def main():
    print("=== 无新到达的纯稳态测试 ===")
    for n in [1, 2, 3, 4, 8, 16]:
        await burst(n)

asyncio.run(main())
'''

for n,c in files.items():
    io.open(n,"w",encoding="utf-8").write(c); print(" ", n, len(c.splitlines()), "行")



## 3. 起服务（开启前缀复用）


In [ ]:
import subprocess, time, requests, os, signal

def serve(extra_args, tag):
    subprocess.run(["pkill","-f","vllm serve"],check=False); time.sleep(6)
    cmd = ["vllm","serve","Qwen/Qwen2.5-0.5B-Instruct","--dtype","half",
           "--max-model-len","2048","--gpu-memory-utilization","0.85",
           "--max-num-seqs","32","--port","8000"] + extra_args
    log = open(f"/content/srv_{tag}.log","w")
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    for i in range(180):
        try:
            if requests.get("http://127.0.0.1:8000/health",timeout=2).status_code==200:
                print(f"[{tag}] 就绪，用时 {i*2}s"); return p
        except Exception: pass
        time.sleep(2)
    print(f"[{tag}] 启动失败，日志尾部：")
    print(open(f"/content/srv_{tag}.log").read()[-1500:]); return None

proc = serve(["--enable-prefix-caching"], "on")


## 4. 并发扫描 + 稳态突发

**对照本地 GTX 1650（无 TC）实测**：稳态突发下并发 1 → 109.5 tok/s，
并发 2 骤降至 **13.3 tok/s**，且墙钟在 batch 2–16 恒定 7.8 s。
T4 若呈现正常的单调上升曲线，即为塌陷源于「无 Tensor Core」的佐证。


In [ ]:
!python -u isolate.py


In [ ]:
!python -u bench_serving.py


## 5. 前缀复用：开


In [ ]:
!python -u bench_serving.py --prefix-test


## 6. 前缀复用：关（受控对照）


In [ ]:
proc = serve([], "off")


In [ ]:
!python -u bench_serving.py --prefix-test


## 7. 结论怎么读

- **第 5 步 TTFT** −  **第 6 步 TTFT** = `--enable-prefix-caching` 的真实收益。
  两次除该开关外配置完全一致，因此差值可归因于该特性。
- 若两者无显著差异，如实记录「本配置下未观察到收益」，
  并说明可能原因（0.5B 模型 prefill 本就便宜，共享前缀仅 2380 字符）。
  **不要因为期望它有效就只报有利的那一组。**
